In [2]:
import pandas as pd

df = pd.read_csv("/content/Hotel_Reviews.csv")
print(df.head())
print(df.info())
print("Dataset Shape:", df.shape)

                                       Hotel_Address  \
0   s Gravesandestraat 55 Oost 1092 AA Amsterdam ...   
1   s Gravesandestraat 55 Oost 1092 AA Amsterdam ...   
2   s Gravesandestraat 55 Oost 1092 AA Amsterdam ...   
3   s Gravesandestraat 55 Oost 1092 AA Amsterdam ...   
4   s Gravesandestraat 55 Oost 1092 AA Amsterdam ...   

   Additional_Number_of_Scoring Review_Date  Average_Score   Hotel_Name  \
0                         194.0    8/3/2017            7.7  Hotel Arena   
1                         194.0    8/3/2017            7.7  Hotel Arena   
2                         194.0   7/31/2017            7.7  Hotel Arena   
3                         194.0   7/31/2017            7.7  Hotel Arena   
4                         194.0   7/24/2017            7.7  Hotel Arena   

  Reviewer_Nationality                                    Negative_Review  \
0              Russia    I am so angry that i made this post available...   
1             Ireland                                     

In [3]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('vader_lexicon')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [5]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):

    text = str(text).lower()

    text = re.sub(r'[^a-zA-Z ]', '', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

df["Clean_Review"] = df["Positive_Review"].apply(preprocess)

print(df[["Positive_Review","Clean_Review"]].head())

                                     Positive_Review  \
0   Only the park outside of the hotel was beauti...   
1   No real complaints the hotel was great great ...   
2   Location was good and staff were ok It is cut...   
3   Great location in nice surroundings the bar a...   
4    Amazing location and building Romantic setting    

                                        Clean_Review  
0                       park outside hotel beautiful  
1  real complaint hotel great great location surr...  
2  location good staff ok cute hotel breakfast ra...  
3  great location nice surroundings bar restauran...  
4         amazing location building romantic setting  


In [8]:
sia = SentimentIntensityAnalyzer()

def sentiment(review):
    review = str(review) # Convert review to string to handle float (NaN) values
    score = sia.polarity_scores(review)["compound"]

    if score >= 0.05:
        return "Positive"

    elif score <= -0.05:
        return "Negative"

    else:
        return "Neutral"

df["Sentiment"] = df["Positive_Review"].apply(sentiment)

print(df[["Positive_Review","Sentiment"]].head())

                                     Positive_Review Sentiment
0   Only the park outside of the hotel was beauti...  Positive
1   No real complaints the hotel was great great ...  Positive
2   Location was good and staff were ok It is cut...  Positive
3   Great location in nice surroundings the bar a...  Positive
4    Amazing location and building Romantic setting   Positive


In [11]:
def service_issue(review):
    review = str(review) # Convert review to string to handle float (NaN) values
    review = review.lower()

    if "room" in review:
        return "Room"

    elif "staff" in review or "reception" in review:
        return "Staff"

    elif "food" in review or "restaurant" in review or "breakfast" in review:
        return "Food"

    elif "clean" in review or "dirty" in review or "housekeeping" in review:
        return "Cleanliness"

    elif "wifi" in review or "pool" in review or "gym" in review or "spa" in review:
        return "Amenities"

    else:
        return "Other"

df["Service_Issue"] = df["Positive_Review"].apply(service_issue)

print(df[["Positive_Review","Service_Issue"]].head())

                                     Positive_Review Service_Issue
0   Only the park outside of the hotel was beauti...         Other
1   No real complaints the hotel was great great ...          Room
2   Location was good and staff were ok It is cut...         Staff
3   Great location in nice surroundings the bar a...          Food
4    Amazing location and building Romantic setting          Other


In [12]:
print("Customer Satisfaction Report\n")

print("Total Reviews :", len(df))

print("\nSentiment Distribution")
print(df["Sentiment"].value_counts())

print("\nService Issues")
print(df["Service_Issue"].value_counts())

positive = (df["Sentiment"] == "Positive").mean() * 100
negative = (df["Sentiment"] == "Negative").mean() * 100
neutral = (df["Sentiment"] == "Neutral").mean() * 100

print("\nPositive Reviews :", round(positive,2), "%")
print("Negative Reviews :", round(negative,2), "%")
print("Neutral Reviews :", round(neutral,2), "%")

Customer Satisfaction Report

Total Reviews : 144395

Sentiment Distribution
Sentiment
Positive    120468
Neutral      21922
Negative      2005
Name: count, dtype: int64

Service Issues
Service_Issue
Other          51570
Room           45434
Staff          34038
Food            8906
Cleanliness     3229
Amenities       1218
Name: count, dtype: int64

Positive Reviews : 83.43 %
Negative Reviews : 1.39 %
Neutral Reviews : 15.18 %


In [13]:
sample_review = "The room was clean and spacious. Staff were friendly and breakfast was delicious."

clean_review = preprocess(sample_review)

print("Clean Review:", clean_review)
print("Sentiment:", sentiment(sample_review))
print("Service Issue:", service_issue(sample_review))

Clean Review: room clean spacious staff friendly breakfast delicious
Sentiment: Positive
Service Issue: Room
